In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import json
import os
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from tqdm import tqdm

# ------------------------------------------------------
# DEVICE SETUP
# ------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == 'cuda':
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True

# ------------------------------------------------------
# PATHS
# ------------------------------------------------------
DATA_ROOT = "../data"
TRAIN_DATA_PARENT = os.path.join(DATA_ROOT, "target_4_December_release")
VAL_DATA_PARENT = os.path.join(DATA_ROOT, "cleaned_dev_10_january_2025")
TEST_DATA_PARENT = os.path.join(DATA_ROOT, "testdata_ST12")
TAXONOMY_FILE = os.path.join(DATA_ROOT, "taxonomy.json")

available_languages = [d for d in os.listdir(TRAIN_DATA_PARENT) if os.path.isdir(os.path.join(TRAIN_DATA_PARENT, d))]
print("Detected languages:", available_languages)

# ------------------------------------------------------
# TAXONOMY PARSING
# ------------------------------------------------------
with open(TAXONOMY_FILE, "r", encoding="utf-8") as f:
    taxonomy = json.load(f)

coarse_roles, fine_roles, role_to_parent = [], [], {}

for i, category in enumerate(taxonomy):
    main_name = category["name"]
    coarse_roles.append(f"{main_name}: {category['description'] if 'description' in category else ''}")
    for subtype in category["subtypes"]:
        fine_text = f"{subtype['name']}: {subtype['description']}. Example: {subtype['example']}"
        fine_roles.append(fine_text)
        role_to_parent[len(fine_roles) - 1] = i  # fine index → coarse index

print(f"\n✅ Loaded taxonomy with {len(coarse_roles)} coarse and {len(fine_roles)} fine roles.")

main_categories = [cat["name"] for cat in taxonomy]
label2id = {label["name"] if isinstance(label, dict) else label: i for i, label in enumerate(main_categories)}

# ------------------------------------------------------
# LOAD ANNOTATIONS (same as your version)
# ------------------------------------------------------
def load_annotations(annotation_path, docs_root, labeled=True):
    data = []
    with open(annotation_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if labeled:
                if len(parts) < 5:
                    continue
                doc_id, mention, start, end, *labels = parts
                label = labels[0]
            else:
                if len(parts) < 4:
                    continue
                doc_id, mention, start, end = parts
                label = None

            start, end = int(start), int(end)
            text_path = os.path.join(docs_root, doc_id)
            if not os.path.exists(text_path):
                continue

            with open(text_path, "r", encoding="utf-8") as doc_file:
                text = doc_file.read()

            left = max(0, start - 100)
            right = min(len(text), end + 100)
            context = text[left:right]

            entry = {
                "doc_id": doc_id,
                "text": context,
                "mention": mention,
                "start": start,
                "end": end,
            }
            if labeled:
                entry["label"] = label
            data.append(entry)
    return pd.DataFrame(data)


def load_multilingual_data(labeled=True):
    all_train, all_val = [], []
    for lang in available_languages:
        print(f"\n📘 Loading data for language: {lang}")
        train_root = os.path.join(TRAIN_DATA_PARENT, lang, "raw-documents")
        train_ann = os.path.join(TRAIN_DATA_PARENT, lang, "subtask-1-annotations.txt")
        val_root = os.path.join(VAL_DATA_PARENT, lang, "subtask-1-documents")
        val_ann = os.path.join(VAL_DATA_PARENT, lang, "subtask-1-annotations.txt")

        if not (os.path.exists(train_ann) and os.path.exists(val_ann)):
            print(f"⚠️ Skipping {lang}: missing files")
            continue

        train_df = load_annotations(train_ann, train_root, labeled=True)
        val_df = load_annotations(val_ann, val_root, labeled=True)
        all_train.append(train_df)
        all_val.append(val_df)

    return pd.concat(all_train, ignore_index=True), pd.concat(all_val, ignore_index=True)


train_df_full, val_df = load_multilingual_data(labeled=True)
train_df, new_val_df = train_test_split(train_df_full, test_size=0.2, random_state=42, stratify=train_df_full['label'])
val_df = new_val_df

print(f"\n✅ Final dataset sizes: Train={len(train_df)}, Val={len(val_df)}")

# ------------------------------------------------------
# DATASET CLASS (adapted)
# ------------------------------------------------------
class EntityFramingDataset(Dataset):
    def __init__(self, df, tokenizer, taxonomy, coarse_names, fine_names, role_to_parent, max_len=256):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

        # mapping
        self.coarse_names = coarse_names
        self.fine_names = [s["name"] for c in taxonomy for s in c["subtypes"]]
        self.role_to_parent = role_to_parent

        self.coarse2id = {name: i for i, name in enumerate(coarse_names)}
        self.fine2id = {name: i for i, name in enumerate(self.fine_names)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["text"]
        mention = row["mention"]
        marked = text.replace(mention, f"[ENTITY] {mention} [/ENTITY]")

        inputs = self.tokenizer(
            marked,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
            return_offsets_mapping=True
        )
        offsets = inputs.pop("offset_mapping")[0]
        start_tok, end_tok = 0, 1
        for j, (a, b) in enumerate(offsets):
            if a <= row["start"] < b:
                start_tok = j
            if a < row["end"] <= b:
                end_tok = j + 1

        coarse_label = torch.zeros(len(self.coarse_names))
        fine_label = torch.zeros(len(self.fine_names))

        # Assign fine and parent category
        if "label" in row and row["label"] in self.coarse2id:
            coarse_label[self.coarse2id[row["label"]]] = 1.0
            # if available: sample subtype or fallback
            fine_label[:] = 0  # placeholder, modify when subtype labels available

        return {
            "input_ids": inputs["input_ids"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "span": torch.tensor([start_tok, end_tok], dtype=torch.long),
            "coarse_label": coarse_label,
            "fine_label": fine_label
        }

# ------------------------------------------------------
# INITIALIZE TOKENIZER & DATASETS
# ------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

train_dataset = EntityFramingDataset(train_df, tokenizer, taxonomy, main_categories, fine_roles, role_to_parent)
val_dataset = EntityFramingDataset(val_df, tokenizer, taxonomy, main_categories, fine_roles, role_to_parent)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

print(f"\n✅ DataLoader initialized: {len(train_loader)} batches.")



Using device: cuda
Detected languages: ['BG', 'EN', 'HI', 'PT', 'RU']

✅ Loaded taxonomy with 3 coarse and 22 fine roles.

📘 Loading data for language: BG

📘 Loading data for language: EN

📘 Loading data for language: HI

📘 Loading data for language: PT

📘 Loading data for language: RU

✅ Final dataset sizes: Train=4209, Val=1053

✅ DataLoader initialized: 1053 batches.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from typing import List, Dict, Tuple, Optional

# ---------------------------------------------------
# Base Role Encoder (trainable)
# ---------------------------------------------------

class RoleEncoder(nn.Module):
    """
    Encodes role definitions into vectors using a transformer backbone.
    Trainable (fine-tuned during task).
    """
    def __init__(self, pretrained_model="xlm-roberta-base", proj_dim=768, freeze_lower_layers: int = 0):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(pretrained_model)
        self.tokenizer = AutoTokenizer.from_pretrained(pretrained_model, use_fast=True)
        self.hidden_size = self.backbone.config.hidden_size
        self.proj = nn.Linear(self.hidden_size, proj_dim)
        self.norm = nn.LayerNorm(proj_dim)
        self.proj_dim = proj_dim
        self.freeze_lower_layers(freeze_lower_layers)

    def freeze_lower_layers(self, n_layers: int):
        if n_layers > 0:
            for i in range(n_layers):
                for param in self.backbone.encoder.layer[i].parameters():
                    param.requires_grad = False

    def forward(self, texts: List[str], device=None):
        enc = self.tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
        input_ids = enc["input_ids"].to(device)
        attn_mask = enc["attention_mask"].to(device)
        out = self.backbone(input_ids=input_ids, attention_mask=attn_mask, return_dict=True)
        pooled = out.last_hidden_state[:, 0, :]  # CLS token
        vec = self.proj(pooled)
        vec = self.norm(vec)
        return vec  # (N, proj_dim)


# ---------------------------------------------------
# Mention Encoder
# ---------------------------------------------------

class MentionEncoder(nn.Module):
    def __init__(self, pretrained_model="xlm-roberta-base", proj_dim=768):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(pretrained_model)
        self.tokenizer = AutoTokenizer.from_pretrained(pretrained_model, use_fast=True)
        self.hidden_size = self.backbone.config.hidden_size
        self.proj = nn.Linear(self.hidden_size * 2, proj_dim)
        self.norm = nn.LayerNorm(proj_dim)
        self.proj_dim = proj_dim

    def forward(self, input_ids, attention_mask, span_indices):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        last_hidden = out.last_hidden_state
        batch_vecs = []
        for i, (s, e) in enumerate(span_indices):
            span_vec = last_hidden[i, s:e, :].mean(dim=0)
            cls_vec = last_hidden[i, 0, :]
            v = torch.cat([cls_vec, span_vec], dim=-1)
            batch_vecs.append(v)
        h = torch.stack(batch_vecs)
        return self.norm(self.proj(h))

    def tokenize_with_markers(self, texts: List[str], spans: List[Tuple[int,int]], max_length=512):
        enc = self.tokenizer(texts, return_offsets_mapping=True, truncation=True, padding=True, max_length=max_length, return_tensors="pt")
        offsets = enc.pop("offset_mapping")
        span_indices = []
        for i, (start_char, end_char) in enumerate(spans):
            offs = offsets[i]
            start_tok, end_tok = 0, 1
            for j, (a, b) in enumerate(offs):
                if a <= start_char < b: start_tok = j
                if a < end_char <= b: end_tok = j + 1
            span_indices.append((start_tok, end_tok))
        return enc["input_ids"], enc["attention_mask"], span_indices


# ---------------------------------------------------
# Hierarchical Dual Encoder (trainable)
# ---------------------------------------------------

class HierarchicalDualEncoder(nn.Module):
    def __init__(
        self,
        pretrained="xlm-roberta-base",
        coarse_role_texts: List[str] = None,
        fine_role_texts: List[str] = None,
        role_to_parent: Dict[int,int] = None,
        proj_dim=768,
        freeze_coarse_layers=6,
        freeze_fine_layers=0,
        temperature_coarse=1.0,
        temperature_fine=1.0,
        device=torch.device("cpu")
    ):
        super().__init__()
        self.device = device
        self.proj_dim = proj_dim
        self.role_to_parent = role_to_parent or {}
        self.C = len(coarse_role_texts or [])
        self.F = len(fine_role_texts or [])

        # Encoders
        self.mention_encoder = MentionEncoder(pretrained, proj_dim)
        self.role_encoder_coarse = RoleEncoder(pretrained, proj_dim, freeze_lower_layers=freeze_coarse_layers)
        self.role_encoder_fine = RoleEncoder(pretrained, proj_dim, freeze_lower_layers=freeze_fine_layers)

        self.coarse_texts = coarse_role_texts
        self.fine_texts = fine_role_texts

        # Temperatures
        self.tau_c = nn.Parameter(torch.tensor(temperature_coarse))
        self.tau_f = nn.Parameter(torch.tensor(temperature_fine))

        self.to(device)

    def cosine(self, a, b):
        a = a / (a.norm(dim=-1, keepdim=True) + 1e-9)
        b = b / (b.norm(dim=-1, keepdim=True) + 1e-9)
        return a @ b.T

    def forward(self, input_ids, attention_mask, span_indices):
        h = self.mention_encoder(input_ids, attention_mask, span_indices)

        # Encode roles dynamically each step (trainable)
        c_vecs = self.role_encoder_coarse(self.coarse_texts, device=self.device)
        f_vecs = self.role_encoder_fine(self.fine_texts, device=self.device)

        sim_c = self.cosine(h, c_vecs) / self.tau_c
        sim_f = self.cosine(h, f_vecs) / self.tau_f

        p_coarse = torch.sigmoid(sim_c)
        p_fine_raw = torch.sigmoid(sim_f)

        # Gating by parent probabilities
        if self.F > 0:
            parent_probs = []
            for i in range(self.F):
                parent = self.role_to_parent.get(i)
                parent_probs.append(p_coarse[:, parent] if parent is not None else torch.ones(h.size(0), device=self.device))
            parent_probs = torch.stack(parent_probs, dim=1)
            p_fine = p_fine_raw * parent_probs
        else:
            p_fine = p_fine_raw

        return p_coarse, p_fine


# ---------------------------------------------------
# Loss functions
# ---------------------------------------------------

def bce_loss(pred, gold):
    return F.binary_cross_entropy(pred, gold.float())

def hier_loss(p_coarse, y_fine, role_to_parent, delta=0.3):
    loss = 0
    B, F = y_fine.shape
    for i in range(F):
        parent = role_to_parent.get(i)
        if parent is not None:
            mask = (y_fine[:, i] == 1) & (p_coarse[:, parent] < delta)
            if mask.any():
                loss += (delta - p_coarse[:, parent][mask]).mean()
    return loss

In [21]:
model = HierarchicalDualEncoder(
    pretrained="roberta-base",
    coarse_role_texts=main_categories,
    fine_role_texts=fine_roles,
    role_to_parent=role_to_parent,
    freeze_coarse_layers=10,
    freeze_fine_layers=10,
    device=device
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# ------------------------------------------------------
# TRAINING LOOP
# ------------------------------------------------------
def train_epoch(model, loader, optimizer, alpha=0.5, beta=0.2):
    model.train()
    total_loss = 0.0
    for batch in tqdm(loader):
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        spans = batch["span"]
        y_coarse = batch["coarse_label"].to(device)
        y_fine = batch["fine_label"].to(device)

        p_c, p_f = model(input_ids, attn, spans)
        loss_c = bce_loss(p_c, y_coarse)
        loss_f = bce_loss(p_f, y_fine)
        loss_h = hier_loss(p_c, y_fine, role_to_parent)
        loss = loss_f + alpha * loss_c + beta * loss_h

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# ------------------------------------------------------
# TRAIN FOR ONE EPOCH (demo)
# ------------------------------------------------------
loss = train_epoch(model, train_loader, optimizer)
print(f"\n✅ Training epoch finished, average loss: {loss:.4f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 5.27 GiB is allocated by PyTorch, and 40.56 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)